In [13]:
!uv pip install --upgrade torch transformers==4.45.0 huggingface_hub accelerate numpy<2.0
!uv pip uninstall torchvision

/bin/bash: line 1: 2.0: No such file or directory
Using Python 3.12.13 environment at: /usr


In [14]:
import os
os.environ["WANDB_DISABLED"] = "true"
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
import math
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_scheduler
)
from accelerate import Accelerator
from accelerate.utils import set_seed
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)
print(f"Torch CUDA: {torch.version.cuda}")

Torch CUDA: 12.8


In [15]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
             return focal_loss.sum()
        return focal_loss

In [19]:
%%writefile train.py
import os
os.environ["WANDB_DISABLED"] = "true"
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
import math
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_scheduler
)
from accelerate import Accelerator
from accelerate.utils import set_seed
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
             return focal_loss.sum()
        return focal_loss

class AcceleratedAdvancedTrainer:
    def __init__(self, max_length=512, model_name="microsoft/graphcodebert-base", seed=42):
        self.max_length = max_length
        self.model_name = model_name
        self.num_labels = None
        self.accelerator = Accelerator(log_with="all")
        set_seed(seed)

        # Remove red error boxes by pushing HuggingFace logs to pure stdout via logger instead of stderr
        logger.setLevel(logging.INFO if self.accelerator.is_main_process else logging.ERROR)
        from transformers.utils import logging as hf_logging
        hf_logging.set_verbosity_info()
        hf_logging.disable_default_handler()
        hf_logging.add_handler(logging.StreamHandler(os.sys.stdout))

    def load_and_prepare_data(self):
        try:
            # Load datasets from Hugging Face Hub
            train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
            val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")

            if 'code' not in train_dataset.column_names or 'label' not in train_dataset.column_names:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            # Determine num_labels from the training dataset
            self.num_labels = len(set(train_dataset['label']))

            logger.info(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")
            return train_dataset, val_dataset
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            raise

    def preprocess_code(self, code_str):
        code_str = re.sub(r'[\r\n]+', '\n', code_str)
        code_str = re.sub(r'[ \t]+', ' ', code_str)
        return code_str

    def initialize_model_and_tokenizer(self):
        logger.info(f"Initializing {self.model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification"
        )

    def tokenize_function(self, examples):
        cleaned_codes = [self.preprocess_code(c) for c in examples['code']]
        return self.tokenizer(
            cleaned_codes,
            padding=True,
            max_length=self.max_length,
        )

    def prepare_dataloaders(self, train_dataset, val_dataset, batch_size): # Modified parameters
        logger.info("Preparing DataLoaders...")

        # Accelerate: Do tokenization and remove unused columns
        with self.accelerator.main_process_first():
            train_dataset = train_dataset.map(self.tokenize_function, batched=True, remove_columns=['code']) # 'code' removed after tokenization
            val_dataset = val_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])

            # Rename 'label' column to 'labels' as expected by the model
            train_dataset = train_dataset.rename_column('label', 'labels')
            val_dataset = val_dataset.rename_column('label', 'labels')

        # Set format for PyTorch
        train_dataset.set_format("torch")
        val_dataset.set_format("torch")

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        train_dataloader = DataLoader(train_dataset, shuffle=True, collate_fn=data_collator, batch_size=batch_size)
        val_dataloader = DataLoader(val_dataset, collate_fn=data_collator, batch_size=batch_size)
        return train_dataloader, val_dataloader

    def train(self, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5, weight_decay=0.01):
        train_dataset, val_dataset = self.load_and_prepare_data() # Modified to accept dataset objects
        self.initialize_model_and_tokenizer()

        train_dataloader, val_dataloader = self.prepare_dataloaders(train_dataset, val_dataset, batch_size)

        # Optimizer
        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_grouped_parameters = [
            {
                "params": [p for n, p in self.model.named_parameters() if not any(nd in n for nd in no_decay)],
                "weight_decay": weight_decay,
            },
            {
                "params": [p for n, p in self.model.named_parameters() if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
            },
        ]
        optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=learning_rate)

        # Scheduler
        num_update_steps_per_epoch = len(train_dataloader)
        max_train_steps = num_epochs * num_update_steps_per_epoch
        lr_scheduler = get_scheduler(
            name="linear", optimizer=optimizer, num_warmup_steps=int(max_train_steps * 0.1), num_training_steps=max_train_steps
        )

        focal_loss_fn = FocalLoss()

        # Prepare everything with accelerator
        self.model, optimizer, train_dataloader, val_dataloader, lr_scheduler = self.accelerator.prepare(
            self.model, optimizer, train_dataloader, val_dataloader, lr_scheduler
        )

        logger.info("***** Running training *****")
        logger.info(f"  Num Epochs = {num_epochs}")
        logger.info(f"  Instantaneous batch size per device = {batch_size}")
        logger.info(f"  Total optimization steps = {max_train_steps}")

        progress_bar = tqdm(range(max_train_steps), disable=not self.accelerator.is_local_main_process)

        best_f1 = 0.0

        for epoch in range(num_epochs):
            self.model.train()
            total_loss = 0

            for step, batch in enumerate(train_dataloader):
                outputs = self.model(**batch)

                # Custom Loss
                loss = focal_loss_fn(outputs.logits, batch["labels"])
                total_loss += loss.detach().float()

                self.accelerator.backward(loss)

                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
                progress_bar.update(1)

            # Evaluation phase
            self.model.eval()
            all_preds = []
            all_labels = []

            for step, batch in enumerate(val_dataloader):
                with torch.no_grad():
                    outputs = self.model(**batch)

                preds = outputs.logits.argmax(dim=-1)

                # Gather across all processes
                preds, labels = self.accelerator.gather_for_metrics((preds, batch["labels"]))
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())

            all_preds = np.concatenate(all_preds)
            all_labels = np.concatenate(all_labels)

            macro_f1 = precision_recall_fscore_support(all_labels, all_preds, average='macro')[2]
            acc = accuracy_score(all_labels, all_preds)

            logger.info(f"Epoch {epoch} | Loss: {total_loss/len(train_dataloader):.4f} | Validation Acc: {acc:.4f} | Macro F1: {macro_f1:.4f}")

            if macro_f1 > best_f1:
                logger.info(f"New best Macro F1 score: {macro_f1:.4f} > {best_f1:.4f}. Saving model...")
                best_f1 = macro_f1

                # Wait before saving
                self.accelerator.wait_for_everyone()
                if self.accelerator.is_main_process:
                    unwrapped_model = self.accelerator.unwrap_model(self.model)
                    unwrapped_model.save_pretrained(output_dir, save_function=self.accelerator.save)
                    self.tokenizer.save_pretrained(output_dir)

        # Print final classification report on main process
        if self.accelerator.is_main_process:
            logger.info("Final Best Validation Model Info:")
            logger.info(f"Best Macro F1: {best_f1:.4f}")
            print(classification_report(all_labels, all_preds, digits=4))
if __name__ == "__main__":
    OUTPUT_DIR = "taskA-graphcodebert-focal-accelerated"

    # Initialize our accelerated trainer
    trainer_obj = AcceleratedAdvancedTrainer(
        max_length=512,
        model_name="microsoft/graphcodebert-base",
        seed=42
    )

    # Start the distributed training run
    trainer_obj.train(
        output_dir=OUTPUT_DIR,
        num_epochs=1,
        batch_size=16, # Adjust dynamically, accelerator spreads this per-device
        learning_rate=2e-5,
        weight_decay=0.01
    )

Overwriting train.py


In [ ]:
# !accelerate launch --dynamo_backend inductor train.py
!python train.py

04/08/2026 16:12:27 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/datasets/DaniilOr/SemEval-2026-Task13/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
04/08/2026 16:12:28 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/DaniilOr/SemEval-2026-Task13/df2aec18238a47fabfb22c79570db15fd1588aa3/README.md "HTTP/1.1 200 OK"
04/08/2026 16:12:28 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/datasets/DaniilOr/SemEval-2026-Task13/resolve/df2aec18238a47fabfb22c79570db15fd1588aa3/SemEval-2026-Task13.py "HTTP/1.1 404 Not Found"
04/08/2026 16:12:28 - INFO - httpx - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/DaniilOr/SemEval-2026-Task13/DaniilOr/SemEval-2026-Task13.py "HTTP/1.1 404 Not Found"
04/08/2026 16:12:28 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/DaniilOr/SemEval-2026-Task13/revision/df2aec18238a47fabfb22c79570db15fd1588aa3 "HTTP/1.1 200 OK"
04/08/2026 

In [18]:
from itertools import chain
from tqdm.auto import tqdm

@torch.no_grad()
def accelerated_predict(trainer_obj, output_path, max_length=512, batch_size=16):
    # Retrieve un-wrapped model or trained model from accelerator directly
    model = trainer_obj.accelerator.unwrap_model(trainer_obj.model)
    tokenizer = trainer_obj.tokenizer

    device = trainer_obj.accelerator.device
    model.to(device)
    model.eval()

    # Load test dataset from Hugging Face Hub
    ds = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="test", streaming=True)
    it = iter(ds)
    first = next(it)
    stream = chain([first], it)

    def batcher(iterator, bs):
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    # Distributed inference is complex over streaming without DDP sharding,
    # so we perform inference on the main device (rank 0).
    if trainer_obj.accelerator.is_main_process:
        with open(output_path, "w") as f:
            f.write("id,label\n")
            logger.info("Predicting over Test Set...")

            for batch in tqdm(batcher(stream, batch_size), desc="Predicting", disable=not trainer_obj.accelerator.is_main_process):
                codes = [trainer_obj.preprocess_code(row["code"]) for row in batch]
                ids   = [row["id"] for row in batch]

                enc = tokenizer(
                    codes,
                    truncation=True,
                    padding=True,
                    max_length=max_length,
                    return_tensors="pt",
                )
                input_ids = enc["input_ids"].to(device)
                attention_mask = enc["attention_mask"].to(device)

                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                pred_labels = logits.argmax(dim=-1).cpu().tolist()

                for ex_id, pred in zip(ids, pred_labels):
                    f.write(f"{ex_id},{pred}\n")

        logger.info(f"Predictions saved to {output_path}")

OUT_CSV = "submission.csv"

accelerated_predict(
    trainer_obj=trainer_obj,
    output_path=OUT_CSV,
    max_length=512,
    batch_size=16,
)

NameError: name 'trainer_obj' is not defined